# 02 - Build fine-tuning data, arm A (tourism-filtered ILCI + Samanantar)
A T4 GPU runtime makes the LaBSE alignment check fast (CPU also works, slower). Downloads ILCI (62 MB) and Samanantar Hindi (1.9 GB), filters tourism-like pairs,
removes everything that overlaps the test sets, and writes train/dev files + a report.

**Before running:** accept the BPCC terms while logged in to Hugging Face:
https://huggingface.co/datasets/ai4bharat/BPCC

In [2]:
# Cell 1: Hugging Face login (BPCC and FLORES+ are gated)
from huggingface_hub import login, whoami
try:
    print("Logged in as", whoami()["name"])
except Exception:
    from getpass import getpass
    login(token=getpass("Paste your Hugging Face token: "))

Logged in as KiyotakaAyanokoji


In [3]:
# Cell 2: get (or update) the code from GitHub
import os
REPO_URL = "https://github.com/prefierolasoledad/TourMT-Hi.git"
WORKDIR = "/content/tourmt-hi"
if not os.path.exists(WORKDIR):
    !git clone {REPO_URL} {WORKDIR}
else:
    !git -C {WORKDIR} fetch -q origin && git -C {WORKDIR} reset -q --hard origin/main
%cd {WORKDIR}
!pip install -r requirements.txt 2>&1 | grep -iE "error|failed" || true
!git log -1 --oneline

Cloning into '/content/tourmt-hi'...
remote: Enumerating objects: 248, done.
remote: Counting objects: 100% (248/248), done.
remote: Compressing objects: 100% (163/163), done.
remote: Total 248 (delta 146), reused 184 (delta 82), pack-reused 0 (from 0)
Receiving objects: 100% (248/248), 2.82 MiB | 12.07 MiB/s, done.
Resolving deltas: 100% (146/146), done.
/content/tourmt-hi
9a50e50 (HEAD -> main, origin/main, origin/HEAD) Enhance 02_build_finetune_data.ipynb with execution outputs and logging details; remove stdout outputs for Hugging Face login and GitHub code retrieval cells. Update build_finetune_data.py to include caching for filtered pairs and progress bar for LaBSE alignment, improving efficiency and user feedback.


In [4]:
# Cell 3: build arm A (about 15-25 minutes: downloading + reading ~10M Samanantar pairs, then LaBSE)
!python src/build_finetune_data.py --config configs/finetune_arm_a.yaml

Building test-overlap index...
README.md: 100% 75.8k/75.8k [00:00<00:00, 3.99MB/s]
Resolving data files: 100% 234/234 [00:00<00:00, 21207.16it/s]
Resolving data files: 100% 229/229 [00:00<00:00, 21587.57it/s]
eng_Latn.jsonl: 100% 423k/423k [00:00<00:00, 9.60MB/s]
eng_Latn.jsonl: 100% 440k/440k [00:00<00:00, 4.11MB/s]
Generating dev split: 997 examples [00:00, 53802.88 examples/s]
Generating devtest split: 1012 examples [00:00, 127746.58 examples/s]
Resolving data files: 100% 234/234 [00:00<00:00, 21493.71it/s]
Resolving data files: 100% 229/229 [00:00<00:00, 20860.84it/s]
hin_Deva.jsonl: 100% 619k/619k [00:00<00:00, 1.89MB/s]
hin_Deva.jsonl: 100% 645k/645k [00:00<00:00, 1.93MB/s]
Generating dev split: 997 examples [00:00, 111109.60 examples/s]
Generating devtest split: 1012 examples [00:00, 120275.30 examples/s]
Resolving data files: 100% 234/234 [00:00<00:00, 21705.71it/s]
Resolving data files: 100% 229/229 [00:00<00:00, 22389.70it/s]
mai_Deva.jsonl: 100% 616k/616k [00:00<00:00, 1.94M

In [5]:
# Cell 4: show 30 random training pairs so we can judge the quality by eye
import csv, random
rows = list(csv.DictReader(open("data/finetune/arm_a/train.tsv", encoding="utf-8"), delimiter="\t"))
for r in random.Random(0).sample(rows, min(30, len(rows))):
    print(f"[{r['source']}] [LaBSE {r.get('labse', '')}] [{r['keywords']}]\n  EN: {r['en']}\n  HI: {r['hi']}")

[ilci] [LaBSE 0.8341] [hill station,station]
  EN: The ride up to the Mount from the foothill town of Abu -Road serves as an apt reminder of the fact that one is fast approaching a charming Hill Station in India, almost an oasis amidst the desert.
  HI: आबू रोड के तराई वाले कस्बे से माउंट तक की चढ़ाई, इस तथ्य का उचित स्मरण कराती है कि भारत में कोई तेजी से एक ऐसे मनोहर पर्वतीय स्थल तक पहुँच रहा है, जो रेगिस्तान के बीच में एक मरूद्वीप है।
[samanantar] [LaBSE 0.8715] [bus,tourist]
  EN: The medical appliance is kept in the tourist bus.
  HI: चिकित्सा अनुप्रयाज्यता पर्यटक बस में रखा है।
[ilci] [LaBSE 0.8389] [architecture,fort,ruins]
  EN: Once known throughout the state for its magnificent architecture, Sinhagad Fort has been ravaged by the travails of time and today stands mostly in ruins.
  HI: एकबार अपने स्थापत्य के लिए पूरे राज्य में जाना जाता था, समय की थपेड़ों ने सिंहगढ किले को नष्ट कर दिया है, और आज अधिकांश अवशेषों में ही खड़ा है।
[ilci] [LaBSE 0.912] [beach,beaches,tour]
  EN: Sta

In [6]:
# Cell 5: put the report into this notebook's output so it can be saved on the laptop (then Ctrl+S)
print("===REPORT-START===")
print(open("results/finetune_data_arm_a.json", encoding="utf-8").read())
print("===REPORT-END===")

===REPORT-START===
{
  "name": "arm_a",
  "config": {
    "name": "arm_a",
    "sources": {
      "ilci": {
        "repo": "ai4bharat/BPCC",
        "file": "ilci/hin_Deva.tsv",
        "revision": "5bde309374f21c39e7c3d505f28ae11f58f2a220"
      },
      "samanantar": {
        "repo": "ai4bharat/samanantar",
        "config": "hi",
        "revision": "ead34c5b22a0354d350e7dfdd6aece7df5a48f37",
        "max_rows": null
      }
    },
    "min_keyword_hits": 2,
    "core_keywords": [
      "hotel",
      "hotels",
      "guest house",
      "guesthouse",
      "hostel",
      "resort",
      "homestay",
      "check-in",
      "temple",
      "temples",
      "mosque",
      "church",
      "gurudwara",
      "monastery",
      "shrine",
      "pilgrim",
      "pilgrims",
      "pilgrimage",
      "ghat",
      "ghats",
      "fort",
      "palace",
      "monument",
      "monuments",
      "museum",
      "heritage",
      "ruins",
      "tomb",
      "mausoleum",
      "tourist",
